# LeetCode 3351 — Sum of Good Subsequences bằng Python

- Link: https://leetcode.com/problems/sum-of-good-subsequences/
- Độ khó: **Hard**
- Chủ đề: **DP theo giá trị kết thúc** (đếm số dãy con + tổng của chúng cùng lúc), **kỹ thuật đóng góp** (trái × phải)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def sumOfGoodSubsequences(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu "dãy con tốt": hai phần tử **liên tiếp trong dãy con** chênh nhau đúng 1 (tăng hoặc giảm).
2. Kỹ thuật DP **mang hai đại lượng**: `cnt[v]` = số dãy con tốt kết thúc bằng giá trị `v`, `sm[v]` = **tổng** các tổng của chúng. Nối thêm `x` vào `c` dãy con làm tổng tăng `x · c`.
3. Dùng **giá trị** (không phải chỉ số) làm khóa để chuyển trạng thái `O(1)` ⇒ `O(n)`.
4. Cách nhìn thứ hai: **đóng góp** của từng phần tử = `x · (số dãy kết thúc tại i) · (số dãy bắt đầu tại i)`.
5. Kiểm chứng bằng vét cạn mọi dãy con.

## 2. Đề bài & ví dụ

Cho mảng `nums`. Dãy con **tốt** là dãy con mà hiệu tuyệt đối giữa **mọi hai phần tử liên tiếp** bằng đúng 1 (dãy con 1 phần tử luôn tốt). Trả về **tổng các phần tử của tất cả dãy con tốt** (các dãy con lấy theo vị trí — hai dãy con cùng giá trị nhưng khác vị trí được tính riêng), modulo `10^9 + 7`.

**Ví dụ 1:** `[1,2,1]` → `14` (`[1],[2],[1],[1,2],[2,1],[1,2,1]`).
**Ví dụ 2:** `[3,4,5]` → `40`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `0 ≤ nums[i] ≤ 10^5`.

In [1]:
from typing import List
from collections import defaultdict
MOD = 10 ** 9 + 7

examples = [([1, 2, 1], 14), ([3, 4, 5], 40)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[1, 2, 1] -> kỳ vọng 14
[3, 4, 5] -> kỳ vọng 40


## 3. Kiến thức nền

### 3.1. Đếm và cộng dồn đồng thời

Nếu có `c` dãy con với tổng các tổng `S`, nối thêm phần tử `x` vào **mỗi** dãy thì được `c` dãy con mới với tổng các tổng `S + x · c`. Đây là mẫu "DP mang theo cặp `(số lượng, tổng)`" rất hay gặp trong các bài "tổng trên mọi dãy con / mọi đường đi".

In [2]:
c, S, x = 3, 10, 5          # 3 dãy con, tổng các tổng 10 (ví dụ [1],[2,3],[4])
print('sau khi nối x =', x, ':', c, 'dãy con, tổng =', S + x * c, '| kiểm tra:', (1 + 5) + (2 + 3 + 5) + (4 + 5))

sau khi nối x = 5 : 3 dãy con, tổng = 25 | kiểm tra: 25


### 3.2. Khóa theo giá trị

Phần tử `x` chỉ nối được sau dãy con kết thúc bằng `x − 1` hoặc `x + 1`. Vậy chỉ cần nhớ **theo giá trị kết thúc**, không cần nhớ vị trí: `cnt[v]`, `sm[v]` gộp mọi dãy con (đã thấy) kết thúc bằng giá trị `v`.

### 3.3. Thứ tự duyệt

Duyệt `nums` từ trái sang phải; khi xử lý `x = nums[i]`, các bảng chỉ chứa dãy con dùng phần tử ở vị trí `< i` ⇒ đúng thứ tự dãy con.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `[1, 2, 1]`:

| bước | `x` | dãy con mới kết thúc tại đây | `c` | tổng mới | `cnt` / `sm` sau bước |
|---|---|---|---|---|---|
| 0 | 1 | `[1]` | 1 | 1 | `cnt[1]=1, sm[1]=1` |
| 1 | 2 | `[2]`, `[1,2]` | `1 + cnt[1] = 2` | `2·2 + sm[1] = 5` | `cnt[2]=2, sm[2]=5` |
| 2 | 1 | `[1]`, `[2,1]`, `[1,2,1]` | `1 + cnt[2] = 3` | `1·3 + sm[2] = 8` | `cnt[1]=4, sm[1]=9` |

Đáp án `= sm[1] + sm[2] = 9 + 5 = 14` ✓.

## 5. Cách 1 — Brute Force: mọi dãy con

In [3]:
def brute(nums):
    n, total = len(nums), 0
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if all(abs(a - b) == 1 for a, b in zip(seq, seq[1:])):
            total += sum(seq)
    return total % MOD

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

14 | kỳ vọng 14
40 | kỳ vọng 40


**Độ phức tạp:** `O(2^n · n)` — chỉ dùng cho `n ≤ 15`.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt từng `x`:

```
c = 1 + cnt[x-1] + cnt[x+1]                 # [x] đứng một mình + nối sau các dãy kết thúc x±1
s = x·c + sm[x-1] + sm[x+1]
cnt[x] += c;  sm[x] += s
```

Đáp án `= Σ_v sm[v]` (hoặc cộng dồn `s` qua mọi bước). Mỗi bước `O(1)` ⇒ `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo chỉ số `O(n²)`

`c[i]`, `s[i]` cho dãy con **kết thúc tại vị trí `i`**: duyệt mọi `j < i` có `|nums[j] − nums[i]| = 1`.

In [4]:
def dp_quadratic(nums):
    n = len(nums)
    c, s = [0] * n, [0] * n
    for i, x in enumerate(nums):
        c[i], s[i] = 1, x
        for j in range(i):
            if abs(nums[j] - x) == 1:
                c[i] += c[j]
                s[i] += s[j] + x * c[j]
        c[i] %= MOD; s[i] %= MOD
    return sum(s) % MOD

for nums, exp in examples:
    print(dp_quadratic(nums), '| kỳ vọng', exp)

14 | kỳ vọng 14
40 | kỳ vọng 40


### 7.2. Cách 3 — DP theo giá trị `O(n)` (lời giải chính)

In [5]:
def dp_value(nums):
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        c = (1 + cnt[x - 1] + cnt[x + 1]) % MOD
        s = (x * c + sm[x - 1] + sm[x + 1]) % MOD
        cnt[x] = (cnt[x] + c) % MOD
        sm[x] = (sm[x] + s) % MOD
    return sum(sm.values()) % MOD

for nums, exp in examples:
    print(dp_value(nums), '| kỳ vọng', exp)

14 | kỳ vọng 14
40 | kỳ vọng 40


### 7.3. Cách 4 — Kỹ thuật đóng góp: trái × phải

- `L[i]` = số dãy con tốt **kết thúc** tại `i` (duyệt trái → phải, khóa theo giá trị như cách 3 nhưng chỉ đếm).
- `R[i]` = số dãy con tốt **bắt đầu** tại `i` (duyệt phải → trái).

Mỗi dãy con tốt chứa `i` được ghép **duy nhất** từ một nửa trái kết thúc tại `i` và một nửa phải bắt đầu tại `i` ⇒ phần tử `i` xuất hiện trong `L[i] · R[i]` dãy con ⇒ đáp án `= Σ nums[i] · L[i] · R[i]`.

In [6]:
def contribution(nums):
    n = len(nums)
    L, R = [0] * n, [0] * n
    cnt = defaultdict(int)
    for i, x in enumerate(nums):
        L[i] = (1 + cnt[x - 1] + cnt[x + 1]) % MOD
        cnt[x] = (cnt[x] + L[i]) % MOD
    cnt = defaultdict(int)
    for i in range(n - 1, -1, -1):
        x = nums[i]
        R[i] = (1 + cnt[x - 1] + cnt[x + 1]) % MOD
        cnt[x] = (cnt[x] + R[i]) % MOD
    return sum(x * L[i] % MOD * R[i] for i, x in enumerate(nums)) % MOD

for nums, exp in examples:
    print(contribution(nums), '| kỳ vọng', exp)

14 | kỳ vọng 14
40 | kỳ vọng 40


Cách 4 tiện khi cần trả lời thêm câu hỏi kiểu "phần tử `i` thuộc bao nhiêu dãy con tốt" — `L[i] · R[i]`.

## 8. TRACE từng bước

Trace cách 3 trên `[2, 1, 2, 3, 2]`.

In [7]:
def trace(nums):
    cnt, sm = defaultdict(int), defaultdict(int)
    for i, x in enumerate(nums):
        c = 1 + cnt[x - 1] + cnt[x + 1]
        s = x * c + sm[x - 1] + sm[x + 1]
        cnt[x] += c; sm[x] += s
        print(f"i={i} x={x}: c = 1 + cnt[{x-1}]({cnt[x-1]}) + cnt[{x+1}]({cnt[x+1]}) = {c}, "
              f"s = {x}·{c} + sm[{x-1}] + sm[{x+1}] = {s} -> cnt = {dict(cnt)}, sm = {dict(sm)}")
    print('đáp án:', sum(sm.values()), '| vét cạn:', brute(nums))

trace([2, 1, 2, 3, 2])

i=0 x=2: c = 1 + cnt[1](0) + cnt[3](0) = 1, s = 2·1 + sm[1] + sm[3] = 2 -> cnt = {1: 0, 3: 0, 2: 1}, sm = {1: 0, 3: 0, 2: 2}
i=1 x=1: c = 1 + cnt[0](0) + cnt[2](1) = 2, s = 1·2 + sm[0] + sm[2] = 4 -> cnt = {1: 2, 3: 0, 2: 1, 0: 0}, sm = {1: 4, 3: 0, 2: 2, 0: 0}
i=2 x=2: c = 1 + cnt[1](2) + cnt[3](0) = 3, s = 2·3 + sm[1] + sm[3] = 10 -> cnt = {1: 2, 3: 0, 2: 4, 0: 0}, sm = {1: 4, 3: 0, 2: 12, 0: 0}
i=3 x=3: c = 1 + cnt[2](4) + cnt[4](0) = 5, s = 3·5 + sm[2] + sm[4] = 27 -> cnt = {1: 2, 3: 5, 2: 4, 0: 0, 4: 0}, sm = {1: 4, 3: 27, 2: 12, 0: 0, 4: 0}
i=4 x=2: c = 1 + cnt[1](2) + cnt[3](5) = 8, s = 2·8 + sm[1] + sm[3] = 47 -> cnt = {1: 2, 3: 5, 2: 12, 0: 0, 4: 0}, sm = {1: 4, 3: 27, 2: 59, 0: 0, 4: 0}
đáp án: 90 | vét cạn: 90


Chú ý ở bước `i = 4`: `cnt[1]`, `cnt[3]` đã gom **mọi** dãy con kết thúc bằng 1 hoặc 3 — kể cả những dãy dài như `[2, 1, 2, 3]`.

## 9. Code nộp LeetCode

In [8]:
class Solution:
    def sumOfGoodSubsequences(self, nums: List[int]) -> int:
        MOD = 10 ** 9 + 7
        M = max(nums) + 2
        cnt = [0] * (M + 1)
        sm = [0] * (M + 1)
        for x in nums:
            lo = x - 1 if x > 0 else M          # cnt[M] luôn 0: thay cho "x - 1 = -1"
            c = (1 + cnt[lo] + cnt[x + 1]) % MOD
            s = (x * c + sm[lo] + sm[x + 1]) % MOD
            cnt[x] = (cnt[x] + c) % MOD
            sm[x] = (sm[x] + s) % MOD
        return sum(sm) % MOD

for nums, exp in examples:
    got = Solution().sumOfGoodSubsequences(nums)
    print(got, 'OK' if got == exp else 'SAI')

14 OK
40 OK


**Cạm bẫy Python:** với `x = 0`, `cnt[x − 1] = cnt[−1]` **không lỗi** mà đọc phần tử cuối mảng! Ở đây ta trỏ sang ô `M` luôn bằng 0 (vì `nums[i] ≤ M − 2`).

## 10. Vì sao thuật toán đúng

**Bất biến:** sau khi xử lý tiền tố `nums[0..i]`, `cnt[v]` = số dãy con tốt của tiền tố có phần tử cuối mang giá trị `v`, `sm[v]` = tổng các tổng của chúng.

- Dãy con tốt kết thúc **tại vị trí `i`** (giá trị `x`) hoặc là `[x]`, hoặc là một dãy con tốt của `nums[0..i−1]` kết thúc bằng `x − 1` hay `x + 1` rồi nối `x`. Hai tập này rời nhau và phủ hết ⇒ `c` đúng; mỗi dãy nối thêm `x` ⇒ `s` đúng.
- Dãy con kết thúc ở vị trí `< i` không đổi ⇒ cộng `c`, `s` vào `cnt[x]`, `sm[x]` giữ bất biến.

Mọi dãy con tốt kết thúc ở đúng một vị trí ⇒ `Σ sm[v]` là đáp án.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP theo chỉ số | `O(n²)` | `O(n)` |
| 3. DP theo giá trị | `O(n)` | `O(max(nums))` hoặc `O(n)` với dict |
| 4. Đóng góp trái × phải | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dp_quadratic': dp_quadratic, 'dp_value': dp_value, 'contribution': contribution,
         'submit': lambda a: Solution().sumOfGoodSubsequences(a)}
edge = [([5], 5), ([0], 0), ([0, 1], 2), ([1, 0], 2), ([2, 2], 4), ([1, 3], 4), ([0, 0, 1], 3)]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums, f(nums))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3351)
cnt_tests = 0
for _ in range(600):
    n = rng.randint(1, 12)
    nums = [rng.randint(0, 5) for _ in range(n)]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
    cnt_tests += 1
print('Khớp vét cạn trên', cnt_tests, 'mảng ngẫu nhiên.')

Khớp vét cạn trên 600 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
N = 10 ** 5
cases = {
    'ngẫu nhiên 0..10^5': [rng.randint(0, 10 ** 5) for _ in range(N)],
    'xen kẽ 1, 2': [1 + i % 2 for i in range(N)],
    'tăng 0..N-1': list(range(N)),
    'giá trị 0..3': [rng.randint(0, 3) for _ in range(N)],
}
for name, nums in cases.items():
    t = time.time(); a = Solution().sumOfGoodSubsequences(nums); t1 = time.time() - t
    t = time.time(); b = contribution(nums); t2 = time.time() - t
    assert a == b
    print(f"{name:20s}: {a:10d} | nộp {t1:.2f}s | đóng góp {t2:.2f}s")

ngẫu nhiên 0..10^5  :  335126963 | nộp 0.05s | đóng góp 0.13s
xen kẽ 1, 2         :  434771702 | nộp 0.04s | đóng góp 0.08s


tăng 0..N-1         :  832150420 | nộp 0.05s | đóng góp 0.11s
giá trị 0..3        :  349042843 | nộp 0.04s | đóng góp 0.08s


Kiểm tra độc lập: với `nums = 0, 1, …, N−1` (tăng 1), dãy con tốt chính là các **đoạn con liên tiếp** (không thể bỏ phần tử ở giữa). Tổng các đoạn con `= Σ i · (i + 1) · (N − i)`.

In [12]:
for N2 in [1, 5, 1000, 10 ** 5]:
    nums = list(range(N2))
    assert Solution().sumOfGoodSubsequences(nums) == sum(i * (i + 1) * (N2 - i) for i in range(N2)) % MOD
print('dãy tăng 1: khớp công thức tổng các đoạn con.')

dãy tăng 1: khớp công thức tổng các đoạn con.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ cho phép tăng 1 (quên `x + 1`)

In [13]:
def wrong_only_increasing(nums):
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        c = 1 + cnt[x - 1]                       # SAI: thiếu cnt[x + 1]
        s = x * c + sm[x - 1]
        cnt[x] += c; sm[x] += s
    return sum(sm.values()) % MOD

print('sai :', wrong_only_increasing([1, 2, 1]), '| đúng:', Solution().sumOfGoodSubsequences([1, 2, 1]))

sai : 7 | đúng: 14


**Vì sao sai:** "hiệu tuyệt đối bằng 1" cho phép cả `[2, 1]` và `[1, 2, 1]`.

### Lỗi 2 — Cộng `x` một lần thay vì `x · c`

In [14]:
def wrong_add_x_once(nums):
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        c = 1 + cnt[x - 1] + cnt[x + 1]
        s = x + sm[x - 1] + sm[x + 1]            # SAI: mỗi dãy mới đều chứa x
        cnt[x] += c; sm[x] += s
    return sum(sm.values()) % MOD

print('sai :', wrong_add_x_once([3, 4, 5]), '| đúng:', Solution().sumOfGoodSubsequences([3, 4, 5]))

sai : 22 | đúng: 40


**Vì sao sai:** với `x = 4`, có hai dãy mới `[4]` và `[3, 4]` — mỗi dãy đều chứa `4`, nên phải cộng `4 · 2`.

### Lỗi 3 — Đếm dãy con theo **giá trị** (khử trùng lặp) thay vì theo vị trí

In [15]:
def wrong_distinct_values(nums):
    seen = set()
    n = len(nums)
    for mask in range(1, 1 << n):
        seq = tuple(nums[i] for i in range(n) if mask >> i & 1)
        if all(abs(a - b) == 1 for a, b in zip(seq, seq[1:])):
            seen.add(seq)                        # SAI: [1] ở vị trí 0 và 2 bị gộp
    return sum(map(sum, seen)) % MOD

print('sai :', wrong_distinct_values([1, 2, 1]), '| đúng:', Solution().sumOfGoodSubsequences([1, 2, 1]))

sai : 13 | đúng: 14


**Vì sao sai:** ví dụ 1 liệt kê `[1]` **hai lần** — dãy con được xác định bởi tập vị trí.

### Lỗi 4 — Gán đè `cnt[x] = c` thay vì cộng dồn `cnt[x] += c`

In [16]:
def wrong_overwrite(nums):
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        c = 1 + cnt[x - 1] + cnt[x + 1]
        s = x * c + sm[x - 1] + sm[x + 1]
        cnt[x] = c; sm[x] = s                    # SAI: xóa mất các dãy kết thúc bằng x trước đó
    return sum(sm.values()) % MOD

print('sai :', wrong_overwrite([1, 2, 1]), '| đúng:', Solution().sumOfGoodSubsequences([1, 2, 1]))

sai : 13 | đúng: 14


**Vì sao sai:** sau bước cuối, dãy `[1]` ở vị trí 0 vẫn là một dãy con tốt kết thúc bằng giá trị 1 — gán đè làm mất nó (và mất đóng góp `1` của nó). Bảng theo giá trị **gộp** mọi vị trí cùng giá trị, nên phải cộng dồn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng trên mọi dãy con thỏa điều kiện giữa các phần tử kề nhau" ⇒ DP theo phần tử cuối, mang `(số lượng, tổng)`.
- Điều kiện chỉ phụ thuộc **giá trị** của phần tử cuối ⇒ khóa theo giá trị, chuyển `O(1)`.
- "Tổng các phần tử" ⇔ "Σ giá trị · số dãy chứa nó" ⇒ kỹ thuật đóng góp trái × phải.

## 15. Biến thể của bài

1. Đếm **số** dãy con tốt (bỏ `sm`).
2. Hiệu đúng bằng `d` thay vì 1 ⇒ dùng `x − d`, `x + d`.
3. Hiệu **tối đa** `d` ⇒ cần tổng theo **khoảng giá trị** ⇒ Fenwick tree.
4. Tổng **bình phương** các tổng ⇒ mang thêm đại lượng `Σ sum²` (`(S + x)² = S² + 2xS + x²`).

In [17]:
# Biến thể 2 và 4
def sum_with_diff(nums, d):
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        if d == 0:
            c = 1 + cnt[x]; s = x * c + sm[x]
        else:
            c = 1 + cnt[x - d] + cnt[x + d]; s = x * c + sm[x - d] + sm[x + d]
        cnt[x] += c; sm[x] += s
    return sum(sm.values())

def sum_of_squares(nums):
    cnt, sm, sq = defaultdict(int), defaultdict(int), defaultdict(int)
    for x in nums:
        c = 1 + cnt[x - 1] + cnt[x + 1]
        S = sm[x - 1] + sm[x + 1]
        Q = sq[x - 1] + sq[x + 1]
        s = x * c + S
        q = Q + 2 * x * S + x * x * c
        cnt[x] += c; sm[x] += s; sq[x] += q
    return sum(sq.values())

def brute_general(nums, ok, f):
    n, total = len(nums), 0
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if all(ok(a, b) for a, b in zip(seq, seq[1:])):
            total += f(seq)
    return total

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(1, 10))]
    d = rng.randint(0, 3)
    assert sum_with_diff(nums, d) == brute_general(nums, lambda a, b: abs(a - b) == d, sum)
    assert sum_of_squares(nums) == brute_general(nums, lambda a, b: abs(a - b) == 1, lambda q: sum(q) ** 2)
print('Biến thể hiệu d và tổng bình phương khớp vét cạn.')

Biến thể hiệu d và tổng bình phương khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi dãy con | `O(2^n n)` | Oracle |
| 2 | DP theo chỉ số | `O(n²)` | Dễ nghĩ |
| 3 | DP theo giá trị | `O(n)` | **Nộp bài** |
| 4 | Đóng góp trái × phải | `O(n)` | Biết thêm số dãy chứa mỗi phần tử |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[1, 1, 1]`.

**Bài 2 (tính tay):** `[1, 2, 3, 2]`.

**Bài 3 (điền TODO):** đếm **số** dãy con tốt có độ dài `≥ 2`.

**Bài 4 (tự viết lại):** biến thể 3 — hiệu **tối đa** `d` (dùng Fenwick tree).

**Bài 5 (mini challenge):** với mỗi `i`, số dãy con tốt **chứa** phần tử `i`.

In [18]:
def count_len2_todo(nums):
    # TODO
    raise NotImplementedError

def sum_with_max_diff_todo(nums, d):
    # TODO
    raise NotImplementedError

def containing_counts_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không có cặp chênh 1. (2) Chỉ dãy độ dài 1. (3) `3`.

**Bài 2.** (1) Chạy bảng như mục 4 với `x = 1, 2, 3, 2`. (2) `c` lần lượt `1, 2, 3, 4`; `s` lần lượt `1, 5, 14, 25`. (3) `1 + 5 + 14 + 25 = 45`.

**Bài 3.** (1) Đếm tổng số dãy (bỏ `sm`). (2) Trừ `n` dãy độ dài 1. (3) Modulo.

**Bài 4.** (1) Nén giá trị. (2) Fenwick cho `cnt` và `sm`, truy vấn đoạn `[x − d, x + d]`. (3) Cẩn thận `d = 0` gồm chính `x`.

**Bài 5.** (1) `L[i] · R[i]` như cách 4. (2) Bỏ modulo khi kiểm tra nhỏ. (3) Vét cạn đối chiếu.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().sumOfGoodSubsequences([1, 1, 1]) == 3
print('Bài 2:', Solution().sumOfGoodSubsequences([1, 2, 3, 2]), '| vét cạn:', brute([1, 2, 3, 2]))

def count_len2(nums):
    cnt = defaultdict(int)
    total = 0
    for x in nums:
        c = 1 + cnt[x - 1] + cnt[x + 1]
        cnt[x] += c
        total += c
    return (total - len(nums)) % MOD

rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(0, 5) for _ in range(rng.randint(1, 10))]
    exp = sum(1 for mask in range(1, 1 << len(nums))
              if bin(mask).count('1') >= 2 and all(abs(a - b) == 1 for a, b in
                  zip(*(lambda q: (q, q[1:]))([nums[i] for i in range(len(nums)) if mask >> i & 1]))))
    assert count_len2(nums) == exp
print('Bài 1: 3 | Bài 3 khớp vét cạn.')

Bài 2: 45 | vét cạn: 45


Bài 1: 3 | Bài 3 khớp vét cạn.


In [20]:
# Bài 4, 5
class BIT:
    def __init__(self, n):
        self.n, self.t = n, [0] * (n + 1)
    def add(self, i, v):
        i += 1
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def pre(self, i):          # tổng [0, i)
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

import bisect
def sum_with_max_diff(nums, d):
    vals = sorted(set(nums))
    bc, bs = BIT(len(vals)), BIT(len(vals))
    total = 0
    for x in nums:
        lo = bisect.bisect_left(vals, x - d); hi = bisect.bisect_right(vals, x + d)
        C = bc.pre(hi) - bc.pre(lo); S = bs.pre(hi) - bs.pre(lo)
        c = 1 + C; s = x * c + S
        p = bisect.bisect_left(vals, x)
        bc.add(p, c); bs.add(p, s)
        total += s
    return total

def containing_counts(nums):
    n = len(nums)
    L, R = [0] * n, [0] * n
    cnt = defaultdict(int)
    for i, x in enumerate(nums):
        L[i] = 1 + cnt[x - 1] + cnt[x + 1]; cnt[x] += L[i]
    cnt = defaultdict(int)
    for i in range(n - 1, -1, -1):
        x = nums[i]; R[i] = 1 + cnt[x - 1] + cnt[x + 1]; cnt[x] += R[i]
    return [L[i] * R[i] for i in range(n)]

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(1, 10))]
    d = rng.randint(0, 3)
    assert sum_with_max_diff(nums, d) == brute_general(nums, lambda a, b: abs(a - b) <= d, sum)
    n = len(nums); exp = [0] * n
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if all(abs(nums[a] - nums[b]) == 1 for a, b in zip(idx, idx[1:])):
            for i in idx:
                exp[i] += 1
    assert containing_counts(nums) == exp
print('Bài 4, 5 khớp vét cạn. Ví dụ containing_counts([1, 2, 1]) =', containing_counts([1, 2, 1]))

Bài 4, 5 khớp vét cạn. Ví dụ containing_counts([1, 2, 1]) = [3, 4, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for x in nums:
    c = 1 + cnt[x-1] + cnt[x+1]
    s = x*c + sm[x-1] + sm[x+1]
    cnt[x] += c; sm[x] += s
return sum(sm) % MOD
```

**Checklist:**

- [ ] Có cả `x − 1` **và** `x + 1`?
- [ ] Cộng `x · c` (không phải `x`)?
- [ ] `+1` cho dãy con một phần tử?
- [ ] Cộng dồn `cnt[x] += c` (không gán đè)?
- [ ] Không đọc `cnt[−1]` khi `x = 0`?
- [ ] Modulo ở mọi phép cộng/nhân?